In [1]:
import pandas as pd

cols = ['대지위치', '법정동코드명', '주용도코드명', '기타용도내용',
        '사용승인일자', '대지면적', '세대수']

df_원본 = pd.read_csv('../data/서울시 건축물대장 표제부.csv',
                      encoding='cp949',
                      encoding_errors='ignore',
                      on_bad_lines='skip',
                      usecols=cols)

# 연립·다세대 + 공동주택 단독 표기(세대수 50 이하) 필터링
mask = (
    df_원본['기타용도내용'].str.contains('연립|다세대', na=False) |
    (
        (df_원본['기타용도내용'] == '공동주택') &
        (df_원본['세대수'] <= 50)
    )
)
df = df_원본[mask].copy()

# 건축연도 추출 및 2022 이하 필터
df['건축연도'] = pd.to_numeric(
    df['사용승인일자'].astype(str).str[:4], errors='coerce')
df = df[df['건축연도'].notna() & (df['건축연도'] <= 2022)].copy()

print(f'최종 건물 수: {len(df):,}')
print(df.head())

최종 건물 수: 95,878
                    대지위치 법정동코드명   대지면적 주용도코드명       기타용도내용  세대수      사용승인일자  \
1   서울특별시 마포구 아현동 751-11    아현동    0.0   공동주택         연립주택  6.0  1984-08-06   
31  서울특별시 양천구 신정동 1023-2    신정동  194.5   공동주택         공동주택  8.0  2002-09-06   
32  서울특별시 양천구 신월동 238-25    신월동    0.0   공동주택        다세대주택  6.0  1990-06-23   
33  서울특별시 양천구 목동 318-186     목동  231.0   공동주택        다세대주택  8.0  1998-06-11   
37   서울특별시 양천구 목동 727-16     목동  138.7   공동주택  공동주택(다세대주택)  5.0  2007-04-30   

      건축연도  
1   1984.0  
31  2002.0  
32  1990.0  
33  1998.0  
37  2007.0  


In [ ]:
import requests
import time
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
KAKAO_KEY = os.getenv('REST_API')

CHECKPOINT = '../data/checkpoint.csv'

def geocode(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {KAKAO_KEY}"}
    params = {"query": address}
    try:
        res = requests.get(url, headers=headers, params=params, timeout=5)
        docs = res.json().get('documents', [])
        if docs:
            return float(docs[0]['x']), float(docs[0]['y'])
        return None, None
    except:
        return None, None

# 이어서 시작
if os.path.exists(CHECKPOINT):
    df_done = pd.read_csv(CHECKPOINT)
    done_indices = set(df_done['idx'])
    print(f'이어서 시작: {len(done_indices):,}건 완료됨')
else:
    df_done = pd.DataFrame(columns=['idx', 'lng', 'lat'])
    done_indices = set()

results = []
total = len(df)

for i, (idx, row) in enumerate(df.iterrows()):
    if idx in done_indices:
        continue

    lng, lat = geocode(row['대지위치'])
    results.append({'idx': idx, 'lng': lng, 'lat': lat})

    if len(results) % 1000 == 0:
        df_done = pd.concat([df_done, pd.DataFrame(results)], ignore_index=True)
        df_done.to_csv(CHECKPOINT, index=False, encoding='utf-8-sig')
        results = []
        elapsed_done = len(done_indices) + len(df_done)
        print(f'{elapsed_done:,} / {total:,} 완료')

    time.sleep(0.05)

# 마지막 남은 것 저장
if results:
    df_done = pd.concat([df_done, pd.DataFrame(results)], ignore_index=True)
    df_done.to_csv(CHECKPOINT, index=False, encoding='utf-8-sig')

print(f'완료. 총 {len(df_done):,}건 저장됨')

/opt/anaconda3/envs/bkh/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


1,000 / 95,878 완료
2,000 / 95,878 완료
3,000 / 95,878 완료
4,000 / 95,878 완료
5,000 / 95,878 완료
6,000 / 95,878 완료
7,000 / 95,878 완료
8,000 / 95,878 완료
9,000 / 95,878 완료
10,000 / 95,878 완료
11,000 / 95,878 완료
12,000 / 95,878 완료
13,000 / 95,878 완료
14,000 / 95,878 완료
15,000 / 95,878 완료
16,000 / 95,878 완료
17,000 / 95,878 완료
18,000 / 95,878 완료
19,000 / 95,878 완료
20,000 / 95,878 완료
21,000 / 95,878 완료
22,000 / 95,878 완료
23,000 / 95,878 완료
24,000 / 95,878 완료
25,000 / 95,878 완료
26,000 / 95,878 완료
27,000 / 95,878 완료
28,000 / 95,878 완료
29,000 / 95,878 완료


In [ ]:
# 집가서 확인용
df_done = pd.read_csv('../data/checkpoint.csv')
print(len(df_done))  # 몇 개까지 저장됐는지 확인